#Instalar la libreria del banco mundial

In [ ]:
!pip install wbdata

#Librerias

In [ ]:
import wbdata
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

#Codigo

In [ ]:
#@title WBPanelBuilder
class WBPanelBuilder:
    def __init__(self, countries, countries_dict, start_year, end_year):
        """
        countries       : lista inicial de códigos WB
        countries_dict  : dict {nombre WB : código WB}
        start_year      : año inicial
        end_year        : año final
        """
        self.countries = countries.copy()
        self.countries_dict = countries_dict.copy()
        self.start_year = str(start_year)
        self.end_year = str(end_year)

        self.base_df = None
        self.indicators_added = []
        self.empty_panel = False  # 🔑 control de estado

    # ---------------------------------------------------
    def _download_indicator(self, indicator_code, indicator_name):
        indicators = {indicator_code: indicator_name}
        return wbdata.get_dataframe(
            indicators,
            country=self.countries,
            date=(self.start_year, self.end_year)
        )

    # ---------------------------------------------------
    def _check_completeness(self, df):
        country_names = df.index.get_level_values(0).unique().tolist()

        pais_completo = []
        pais_incompleto = []

        for country in country_names:
            tmp = df.loc[(country, slice(None))]
            if tmp.isnull().sum().sum() == 0:
                pais_completo.append(country)
            else:
                pais_incompleto.append(country)

        return pais_completo, pais_incompleto

    # ---------------------------------------------------
    def _report_missing(self, df, pais_incompleto):
        for country in pais_incompleto:
            print(f"\n{country}")
            country_data = df.loc[(country, slice(None))]
            for year in country_data.index:
                for col in country_data.columns:
                    if pd.isnull(country_data.loc[year, col]):
                        print(f"  {year}: {col}")

    # ---------------------------------------------------
    def add_indicator(self, indicator_code, indicator_name, report_missing=False):

        # 🚨 Si el panel ya colapsó, no hacemos nada
        if self.empty_panel:
            return

        print(f"\nAgregando indicador: {indicator_name}")

        # 1️⃣ Descarga temporal
        df_temp = self._download_indicator(indicator_code, indicator_name)

        # 2️⃣ Chequeo de completitud
        pais_completo, pais_incompleto = self._check_completeness(df_temp)

        print(f"Paises sin datos faltantes: {len(pais_completo)}")
        print(f"Paises con datos faltantes: {len(pais_incompleto)}")

        if report_missing and pais_incompleto:
            self._report_missing(df_temp, pais_incompleto)

        # 🚨 Si ningún país tiene datos completos → panel vacío
        if len(pais_completo) == 0:
            self.base_df = pd.DataFrame()
            self.countries = []
            self.empty_panel = True
            return

        # 3️⃣ Actualizar países usando el diccionario
        self.countries = [
            self.countries_dict[name] for name in pais_completo
        ]

        # 4️⃣ Redescarga limpia
        df_clean = self._download_indicator(indicator_code, indicator_name)

        # 5️⃣ Merge
        if self.base_df is None:
            self.base_df = df_clean
        else:
            self.base_df = self.base_df.merge(
                df_clean,
                left_index=True,
                right_index=True,
                how="inner"
            )

            # 🚨 Merge vacío → panel colapsa
            if self.base_df.empty:
                self.empty_panel = True

        self.indicators_added.append(indicator_name)

    # ---------------------------------------------------
    def get_data(self):
        if self.base_df is None or self.base_df.empty:
            return pd.DataFrame()
        return self.base_df.copy()

    # ---------------------------------------------------
    def get_number_of_countries(self):
        """
        Devuelve el número de países que tienen información completa
        para todos los indicadores agregados hasta el momento.
        """
        return len(self.countries)


In [ ]:
#@title WBPanelLoop
class WBPanelLoop:
    def __init__(
        self,
        countries,
        countries_dict,
        indicators,
        start_year_range,
        end_year_range
    ):
        """
        countries          : lista inicial de países (WB codes)
        countries_dict     : dict {nombre WB : código WB}
        indicators         : lista de pares [code, name]
        start_year_range   : range de años iniciales
        end_year_range     : range de años finales
        """
        self.countries = countries
        self.countries_dict = countries_dict
        self.indicators = indicators
        self.start_year_range = start_year_range
        self.end_year_range = end_year_range

        self.results_df = None

    # ---------------------------------------------------
    # Corre el loop completo
    # ---------------------------------------------------
    def run(self):
        data = {
            'fecha_inicio': [],
            'fecha_final': [],
            'paises_inicio': [],
            'paises_final': []
        }

        for fecha_empieza in self.start_year_range:
            for fecha_termina in self.end_year_range:

                print(f"\nPeriodo = {fecha_empieza}-{fecha_termina}")

                builder = WBPanelBuilder(
                    countries=self.countries,
                    countries_dict=self.countries_dict,
                    start_year=fecha_empieza,
                    end_year=fecha_termina
                )

                for idx, (code, name) in enumerate(self.indicators):
                    builder.add_indicator(
                        code,
                        name,
                        report_missing=False
                    )

                    if idx == 0:
                        paises_inicio = builder.get_number_of_countries()

                    if idx == len(self.indicators) - 1:
                        paises_final = builder.get_number_of_countries()

                data['fecha_inicio'].append(fecha_empieza)
                data['fecha_final'].append(fecha_termina)
                data['paises_inicio'].append(paises_inicio)
                data['paises_final'].append(paises_final)

        df = pd.DataFrame(data)
        df['numero_anos'] = df['fecha_final'] - df['fecha_inicio'] + 1
        df['paises_perdidos'] = df['paises_inicio'] - df['paises_final']
        df['diferencia']=df['numero_anos'].max()*df['paises_inicio'].max()-df['paises_final']*df['numero_anos']
        df=df.sort_values(by='diferencia')

        self.results_df = df

    # ---------------------------------------------------
    # Extraer DataFrame
    # ---------------------------------------------------
    def get_dataframe(self):
        if self.results_df is None:
            raise ValueError("Primero debes ejecutar run()")
        return self.results_df.copy()

    # ---------------------------------------------------
    # Gráfica
    # ---------------------------------------------------

    def save_scatter(self, title, save_path):
        if self.results_df is None:
            raise ValueError("Primero debes ejecutar run().")

        plt.figure()
        plt.scatter(
            self.results_df['numero_anos'],
            self.results_df['paises_final']
        )
        plt.xlabel('numero_anos')
        plt.ylabel('paises_final')
        plt.title(title)

        plt.savefig(save_path, bbox_inches="tight")
        plt.close()

    # ---------------------------------------------------
    # Solo graficar
    # ---------------------------------------------------
    def plot(self, x='numero_anos', y='paises_final'):
        if self.results_df is None:
            raise ValueError("Primero debes ejecutar run()")

        plt.figure()
        plt.scatter(self.results_df[x], self.results_df[y])
        plt.xlabel(x)
        plt.ylabel(y)
        plt.show()


#Fuentes del banco mundial

In [ ]:
wbdata.get_sources()

  id  name
----  --------------------------------------------------------------------
   1  Doing Business
   2  World Development Indicators
   3  Worldwide Governance Indicators
   5  Subnational Malnutrition Database
   6  International Debt Statistics
  11  Africa Development Indicators
  12  Education Statistics
  13  Enterprise Surveys
  14  Gender Statistics
  15  Global Economic Monitor
  16  Health Nutrition and Population Statistics
  18  IDA Results Measurement System
  19  Millennium Development Goals
  20  Quarterly Public Sector Debt
  22  Quarterly External Debt Statistics SDDS
  23  Quarterly External Debt Statistics GDDS
  25  Jobs
  27  Global Economic Prospects
  28  Global Findex database
  29  The Atlas of Social Protection: Indicators of Resilience and Equity
  30  Exporter Dynamics Database – Indicators at Country-Year Level
  31  Country Policy and Institutional Assessment
  32  Global Financial Development
  33  G20 Financial Inclusion Indicators
  34  Global P

#Paises

In [ ]:
wbdata.get_incomelevels()

id    value
----  -------------------
HIC   High income
INX   Not classified
LIC   Low income
LMC   Lower middle income
LMY   Low & middle income
MIC   Middle income
UMC   Upper middle income

##Paises con ingreso alto

In [ ]:
countries = [i['id'] for i in wbdata.get_countries(incomelevel='HIC')]

##Paises con ingreso medio

In [ ]:
countries.extend([i['id'] for i in wbdata.get_countries(incomelevel='MIC')])

##Paises con ingreso bajo

In [ ]:
countries.extend([i['id'] for i in wbdata.get_countries(incomelevel='LIC')])

In [ ]:
print(len(countries))

215


In [ ]:
#Genero un diccionario con  la lista de countries
countries_dict={wbdata.get_countries(country_id=i)[0]['name']: i for i in countries}

#Indicadores

In [ ]:
wbdata.get_indicators(query="expenditure", source=2)

id                    name
--------------------  -----------------------------------------------------------------------------------------------------------------------------
GB.XPD.RSDV.GD.ZS     Research and development expenditure (% of GDP)
GF.XPD.BUDG.ZS        Primary government expenditures as a proportion of original approved budget (%)
MS.MIL.XPND.CD        Military expenditure (current USD)
MS.MIL.XPND.CN        Military expenditure (current LCU)
MS.MIL.XPND.GD.ZS     Military expenditure (% of GDP)
MS.MIL.XPND.ZS        Military expenditure (% of general government expenditure)
NE.CON.GOVT.CD        General government final consumption expenditure (current US$)
NE.CON.GOVT.CN        General government final consumption expenditure (current LCU)
NE.CON.GOVT.KD        General government final consumption expenditure (constant 2015 US$)
NE.CON.GOVT.KD.ZG     General government final consumption expenditure (annual % growth)
NE.CON.GOVT.KN        General government final consump

In [ ]:
iden_indi=[]

##GDP (constant LCU)

In [ ]:
iden_indi.append(["NY.GDP.MKTP.KN","GDP_cnstn"])
wbdata.get_indicators(indicator="NY.GDP.MKTP.KN")

id              name
--------------  ------------------
NY.GDP.MKTP.KN  GDP (constant LCU)

##GDP growth (annual %)

In [ ]:
iden_indi.append(["NY.GDP.MKTP.KD.ZG","GDP_growth"])
wbdata.get_indicators(indicator="NY.GDP.MKTP.KD.ZG")

id                 name
-----------------  ---------------------
NY.GDP.MKTP.KD.ZG  GDP growth (annual %)

##GDP (constant 2015 US$)

In [ ]:
iden_indi.append(["NY.GDP.MKTP.KD","GDP_cnstn_usd"])
wbdata.get_indicators(indicator="NY.GDP.MKTP.KD")

id              name
--------------  -----------------------
NY.GDP.MKTP.KD  GDP (constant 2015 US$)

##GDP per capita (constant 2015 US$)

In [ ]:
iden_indi.append(["NY.GDP.PCAP.KD","GDP_pp_usd"])
wbdata.get_indicators(indicator="NY.GDP.PCAP.KD")

id              name
--------------  ----------------------------------
NY.GDP.PCAP.KD  GDP per capita (constant 2015 US$)

##GDP per capita (constant LCU)

In [ ]:
iden_indi.append(["NY.GDP.PCAP.KN","GDP_pp_LCU"])
wbdata.get_indicators(indicator="NY.GDP.PCAP.KN")

id              name
--------------  -----------------------------
NY.GDP.PCAP.KN  GDP per capita (constant LCU)

##Inflation, GDP deflator (annual %)

In [ ]:
iden_indi.append(["NY.GDP.DEFL.KD.ZG","Inflatn"])
wbdata.get_indicators(indicator="NY.GDP.DEFL.KD.ZG")

id                 name
-----------------  ----------------------------------
NY.GDP.DEFL.KD.ZG  Inflation, GDP deflator (annual %)

##Foreign direct investment, net inflows (BoP, current US$)

In [ ]:
iden_indi.append(["BX.KLT.DINV.CD.WD","forgn_invst"])
wbdata.get_indicators(indicator="BX.KLT.DINV.CD.WD")

id                 name
-----------------  ---------------------------------------------------------
BX.KLT.DINV.CD.WD  Foreign direct investment, net inflows (BoP, current US$)

##Foreign direct investment, net inflows (% of GDP)

In [ ]:
iden_indi.append(["BX.KLT.DINV.WD.GD.ZS","forgn_invst_gdp"])
wbdata.get_indicators(indicator="BX.KLT.DINV.WD.GD.ZS")

id                    name
--------------------  -------------------------------------------------
BX.KLT.DINV.WD.GD.ZS  Foreign direct investment, net inflows (% of GDP)

##Final consumption expenditure (% of GDP)

In [ ]:
iden_indi.append(["NE.CON.TOTL.ZS","Fin_consum"])
wbdata.get_indicators(indicator="NE.CON.TOTL.ZS")

id              name
--------------  ----------------------------------------
NE.CON.TOTL.ZS  Final consumption expenditure (% of GDP)

##General government final consumption expenditure (% of GDP)

In [ ]:
iden_indi.append(["NE.CON.GOVT.ZS","Fin_consum_govern"])
wbdata.get_indicators(indicator="NE.CON.GOVT.ZS")

id              name
--------------  -----------------------------------------------------------
NE.CON.GOVT.ZS  General government final consumption expenditure (% of GDP)

##External balance on goods and services (% of GDP)

In [ ]:
iden_indi.append(["NE.RSB.GNFS.ZS","Ext_balnce_g_s"])
wbdata.get_indicators(indicator="NE.RSB.GNFS.ZS")

id              name
--------------  -------------------------------------------------
NE.RSB.GNFS.ZS  External balance on goods and services (% of GDP)

##Households and NPISHs final consumption expenditure (% of GDP)

In [ ]:
iden_indi.append(["NE.CON.PRVT.ZS","Fin_consum_hous"])
wbdata.get_indicators(indicator="NE.CON.PRVT.ZS")

id              name
--------------  --------------------------------------------------------------
NE.CON.PRVT.ZS  Households and NPISHs final consumption expenditure (% of GDP)

##Unemployment, total (% of total labor force) (national estimate)


In [ ]:
iden_indi.append(["SL.UEM.TOTL.NE.ZS","Unmploy"])
wbdata.get_indicators(indicator="SL.UEM.TOTL.NE.ZS")

id                 name
-----------------  ----------------------------------------------------------------
SL.UEM.TOTL.NE.ZS  Unemployment, total (% of total labor force) (national estimate)

##Net domestic credit (current LCU)

In [ ]:
#iden_indi.append(["FM.AST.DOMS.CN","dom_credit"])
#wbdata.get_indicators(indicator="FM.AST.DOMS.CN")

##Labor force participation rate, total (% of total population ages 15+) (national estimate)

In [ ]:
iden_indi.append(["SL.TLF.CACT.NE.ZS","Labor_part"])
wbdata.get_indicators(indicator="SL.TLF.CACT.NE.ZS")

id                 name
-----------------  ------------------------------------------------------------------------------------------
SL.TLF.CACT.NE.ZS  Labor force participation rate, total (% of total population ages 15+) (national estimate)

##Portfolio Investment, net (BoP, current US$)

In [ ]:
#iden_indi.append(["BN.KLT.PTXL.CD","port_invst"])
#wbdata.get_indicators(indicator="BN.KLT.PTXL.CD")

#Adicionales

##GDP (current LCU)

In [ ]:
#iden_indi.append(["NY.GDP.MKTP.CN","GDP_nominal"])
#wbdata.get_indicators(indicator="NY.GDP.MKTP.CN")

##GDP (current US$)

In [ ]:
iden_indi.append(["NY.GDP.MKTP.CD","GDP_usd"])
wbdata.get_indicators(indicator="NY.GDP.MKTP.CD")

id              name
--------------  -----------------
NY.GDP.MKTP.CD  GDP (current US$)

#Base de datos

In [ ]:
nam_graf=[]
for i in range(len(iden_indi), len(iden_indi) + 1):
  seleccion = iden_indi[:i]

  loop = WBPanelLoop(
  countries=countries,
  countries_dict=countries_dict,
  indicators=seleccion,
  start_year_range=range(1982, 1984),
  end_year_range=range(2020, 2022)
  )
  #try:
  loop.run()

  df_name = f'{seleccion[-1][-1]}'
  df_resultados = loop.get_dataframe()
  globals()[df_name] = df_resultados

  loop.save_scatter(
    title=f"{seleccion[-1][-1]}",
    save_path=f"{seleccion[-1][-1]}.png"
  )
  nam_graf.append(seleccion[-1][-1])

  #except:
  #  continue



Periodo = 1982-2020

Agregando indicador: GDP_cnstn
Paises sin datos faltantes: 161
Paises con datos faltantes: 54

Agregando indicador: GDP_growth
Paises sin datos faltantes: 159
Paises con datos faltantes: 2

Agregando indicador: GDP_cnstn_usd
Paises sin datos faltantes: 159
Paises con datos faltantes: 0

Agregando indicador: GDP_pp_usd
Paises sin datos faltantes: 159
Paises con datos faltantes: 0

Agregando indicador: GDP_pp_LCU
Paises sin datos faltantes: 159
Paises con datos faltantes: 0

Agregando indicador: Inflatn
Paises sin datos faltantes: 158
Paises con datos faltantes: 1

Agregando indicador: forgn_invst
Paises sin datos faltantes: 139
Paises con datos faltantes: 19

Agregando indicador: forgn_invst_gdp
Paises sin datos faltantes: 139
Paises con datos faltantes: 0

Agregando indicador: Fin_consum
Paises sin datos faltantes: 97
Paises con datos faltantes: 42

Agregando indicador: Fin_consum_govern
Paises sin datos faltantes: 94
Paises con datos faltantes: 3

Agregando indic

In [ ]:
#Mejor periodo
mej_inicio=1983
mej_final=2022

In [ ]:
builder = WBPanelBuilder(
countries=countries,
countries_dict=countries_dict,
start_year=mej_inicio,
end_year=mej_final
)

for x,y in iden_indi:
  builder.add_indicator(
  x,
  y,
  report_missing=False
  )


Agregando indicador: GDP_cnstn
Paises sin datos faltantes: 161
Paises con datos faltantes: 54

Agregando indicador: GDP_growth
Paises sin datos faltantes: 161
Paises con datos faltantes: 0

Agregando indicador: GDP_cnstn_usd
Paises sin datos faltantes: 161
Paises con datos faltantes: 0

Agregando indicador: GDP_pp_usd
Paises sin datos faltantes: 161
Paises con datos faltantes: 0

Agregando indicador: GDP_pp_LCU
Paises sin datos faltantes: 161
Paises con datos faltantes: 0

Agregando indicador: Inflatn
Paises sin datos faltantes: 160
Paises con datos faltantes: 1

Agregando indicador: forgn_invst
Paises sin datos faltantes: 141
Paises con datos faltantes: 19

Agregando indicador: forgn_invst_gdp
Paises sin datos faltantes: 141
Paises con datos faltantes: 0

Agregando indicador: Fin_consum
Paises sin datos faltantes: 99
Paises con datos faltantes: 42

Agregando indicador: Fin_consum_govern
Paises sin datos faltantes: 95
Paises con datos faltantes: 4

Agregando indicador: Ext_balnce_g_s


In [ ]:
panel_final = builder.get_data()
panel_final

GDP_cnstn  GDP_growth  GDP_cnstn_usd    GDP_pp_usd  \
country       date                                                          
Australia     2022  2.552248e+12    4.253046   1.592358e+12  61200.458896   
              2021  2.448128e+12    2.006948   1.527397e+12  59465.535671   
              2020  2.399962e+12   -0.133532   1.497346e+12  58377.767052   
              2019  2.403171e+12    2.194017   1.499348e+12  59181.299790   
              2018  2.351577e+12    2.874170   1.467158e+12  58772.706315   
...                          ...         ...            ...           ...   
United States 1987  9.627608e+12    3.454630   8.849219e+12  36523.404486   
              1986  9.306116e+12    3.462655   8.553720e+12  35620.759359   
              1985  8.994662e+12    4.169575   8.267447e+12  34748.266874   
              1984  8.634635e+12    7.236453   7.936527e+12  33654.307930   
              1983  8.051958e+12    4.583791   7.400960e+12  31656.173660   

                      GDP_pp_LCU   Inflatn   forgn_invst  forgn_invst_gdp  \
country       date                                                          
Australia     2022  98092.754060  7.199544  6.870942e+10         4.052153   
              2021  95312.000446  3.060670  3.145306e+10         2.015424   
              2020  93568.513198  1.865834  1.797684e+10         1.348260   
              2019  94856.424118  3.476600  3.874513e+10         2.770776   
              2018  94201.526099  1.846176  6.068664e+10         4.234508   
...                          ...       ...           ...              ...   
United States 1987  39736.051302  2.477388  6.323500e+10         1.302414   
              1986  38754.008320  2.013893  3.094600e+10         0.675731   
              1985  37804.770246  3.162525  9.630000e+09         0.221942   
              1984  36614.585231  3.607879  2.523000e+10         0.624874   
              1983  34440.692436  3.916909  1.150000e+10         0.316452   

                    Fin_consum  Fin_consum_govern  Ext_balnce_g_s  \
country       date                                                  
Australia     2022   71.071701          22.017954        5.228418   
              2021   73.318371          22.366947        3.801393   
              2020   74.278610          21.813861        3.376511   
              2019   74.579592          20.289384        2.123277   
              2018   75.393593          19.942302        0.051922   
...                        ...                ...             ...   
United States 1987   79.361511          16.001207       -2.981742   
              1986   79.138843          16.114595       -2.879468   
              1985   78.439421          15.917109       -2.627761   
              1984   77.448433          15.720377       -2.544251   
              1983   79.168077          16.383786       -1.421064   

                    Fin_consum_hous  Unmploy  Labor_part       GDP_usd  
country       date                                                      
Australia     2022        49.053747    3.728      66.846  1.695628e+12  
              2021        50.951424    5.022      66.056  1.560617e+12  
              2020        52.464749    6.394      64.980  1.333336e+12  
              2019        54.290208    5.143      66.052  1.398350e+12  
              2018        55.451291    5.345      65.746  1.433145e+12  
...                             ...      ...         ...           ...  
United States 1987        63.360304    6.200      65.600  4.855215e+12  
              1986        63.024248    7.000      65.300  4.579631e+12  
              1985        62.522312    7.200      64.800  4.338979e+12  
              1984        61.728056    7.500      64.400  4.037613e+12  
              1983        62.784291    9.600      64.000  3.634038e+12  

[680 rows x 15 columns]

##Paises considerados

In [ ]:
#Lista de paises considerados
list_countries = panel_final.index.get_level_values(0).unique().tolist()
for i in list_countries:
  print(i)
print(f"En total hay {len(list_countries)} paises")

Australia
Austria
Belgium
Canada
Germany
Denmark
Spain
France
United Kingdom
Greece
Hong Kong SAR, China
Ireland
Italy
Japan
Korea, Rep.
Norway
United States
En total hay 17 paises


#Exportar base de datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Crear carpeta si no existe
folder_path = '/content/drive/MyDrive/Colab Notebooks/Tesis'
os.makedirs(folder_path, exist_ok=True)


In [ ]:
file_path = os.path.join(folder_path, 'base_datos.csv')
panel_final.to_csv(file_path)  # index=False evita guardar el índice
print(f"Guardado en: {file_path}")


Guardado en: /content/drive/MyDrive/Colab Notebooks/Tesis/base_datos.csv


In [ ]:
#Para comprobar llamo a la base
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Tesis/base_datos.csv')
df

,country,date,GDP_cnstn,GDP_growth,GDP_cnstn_usd,GDP_pp_usd,GDP_pp_LCU,Inflatn,forgn_invst,forgn_invst_gdp,Fin_consum,Fin_consum_govern,Ext_balnce_g_s,Fin_consum_hous,Unmploy,Labor_part,GDP_usd
0,Australia,2022,2.552248e+12,4.253046,1.592358e+12,61200.458896,98092.754060,7.199544,6.870942e+10,4.052153,71.071701,22.017954,5.228418,49.053747,3.728,66.846,1.695628e+12
1,Australia,2021,2.448128e+12,2.006948,1.527397e+12,59465.535671,95312.000446,3.060670,3.145306e+10,2.015424,73.318371,22.366947,3.801393,50.951424,5.022,66.056,1.560617e+12
2,Australia,2020,2.399962e+12,-0.133532,1.497346e+12,58377.767052,93568.513198,1.865834,1.797684e+10,1.348260,74.278610,21.813861,3.376511,52.464749,6.394,64.980,1.333336e+12
3,Australia,2019,2.403171e+12,2.194017,1.499348e+12,59181.299790,94856.424118,3.476600,3.874513e+10,2.770776,74.579592,20.289384,2.123277,54.290208,5.143,66.052,1.398350e+12
4,Australia,2018,2.351577e+12,2.874170,1.467158e+12,58772.706315,94201.526099,1.846176,6.068664e+10,4.234508,75.393593,19.942302,0.051922,55.451291,5.345,65.746,1.433145e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,United States,1987,9.627608e+12,3.454630,8.849219e+12,36523.404486,39736.051302,2.477388,6.323500e+10,1.302414,79.361511,16.001207,-2.981742,63.360304,6.200,65.600,4.855215e+12
676,United States,1986,9.306116e+12,3.462655,8.553720e+12,35620.759359,38754.008320,2.013893,3.094600e+10,0.675731,79.138843,16.114595,-2.879468,63.024248,7.000,65.300,4.579631e+12
677,United States,1985,8.994662e+12,4.169575,8.267447e+12,34748.266874,37804.770246,3.162525,9.630000e+09,0.221942,78.439421,15.917109,-2.627761,62.522312,7.200,64.800,4.338979e+12
678,United States,1984,8.634635e+12,7.236453,7.936527e+12,33654.307930,36614.585231,3.607879,2.523000e+10,0.624874,77.448433,15.720377,-2.544251,61.728056,7.500,64.400,4.037613e+12
